In [34]:
import importlib
import tisd_engine
importlib.reload(tisd_engine) # This forces the notebook to read the file again
from tisd_engine import chat_with_tisd

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

In [35]:
# Cell 1
import json
import pandas as pd
from tisd_engine import chat_with_tisd

# Define the test path
TEST_DATA_FILE = "../data/test_set.json"

# Load our test set
with open(TEST_DATA_FILE, "r") as f:
    test_questions = json.load(f)

print(f"Loaded {len(test_questions)} questions for evaluation.")

Loaded 3 questions for evaluation.


In [36]:
# Cell 2
def calculate_similarity(pred, target):
    """Measures how many words overlap between prediction and ground truth."""
    pred_set = set(pred.lower().split())
    target_set = set(target.lower().split())
    
    if len(target_set) == 0: return 0
    
    intersection = pred_set.intersection(target_set)
    return len(intersection) / len(target_set)

def run_evaluation():
    results = []
    print("Running Evaluation Loop...")
    
    for item in test_questions:
        q = item["question"]
        gt = item["ground_truth"]
        
        # Call the engine
        answer, context, latency = chat_with_tisd(q)
        
        # Calculate overlap
        score = calculate_similarity(answer, gt)
        
        results.append({
            "question": q,
            "answer": answer,
            "score": score
        })
        
    return pd.DataFrame(results)

# Run it
df_results = run_evaluation()

Both `max_new_tokens` (=200) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Running Evaluation Loop...


Both `max_new_tokens` (=200) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=200) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


In [37]:
# Cell 3
# Calculate the average performance
avg_score = df_results['score'].mean()

print(f"--- EVALUATION SUMMARY ---")
print(f"Average Accuracy Score: {avg_score:.2%}")
print("-" * 30)

# Display the table cleanly
from IPython.display import display
display(df_results)

--- EVALUATION SUMMARY ---
Average Accuracy Score: 76.82%
------------------------------


,question,answer,score
0,What is the sun?,<|system|>\nYou are a teacher for grade 1-4 st...,0.923077
1,What do plants need to grow?,<|system|>\nYou are a teacher for grade 1-4 st...,0.750000
2,What are sense organs?,<|system|>\nYou are a teacher for grade 1-4 st...,0.631579


In [38]:
# Cell 4
# Show questions where the AI scored less than 30%
low_performers = df_results[df_results['score'] < 0.3]

print(f"Questions needing improvement:")
display(low_performers)

Questions needing improvement:


,question,answer,score
